In [1]:
# using the powerplant dataset:

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("Powerplant dataset.csv")

In [4]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


- AT - Temperature value
- V - Vaccum
- AP - Pressure
- RH - Humidity

- PR - Produced energy (output)

In [5]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [6]:
X = df.drop(columns=["PE"], axis=1)
y= df["PE"]

# Steps that we will follow:
1. Load the data
2. Data tranformation to tensor
3. Tensor dataset & Dataloader
4. Define ANN Model
5. Train the model

In [7]:
# split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
#appplying the standard scaler:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [9]:
# Conversion of data set to Tensors

# tensors are core data structres of pytorch

import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled , dtype=torch.float32)

 # Using.view(-1 , 1) to convert the 1D shape, here -1 means to automatically detect the rows, 1 represents the number of columns:                               
y_train_tensor = torch.tensor(y_train.values , dtype=torch.float32).view(-1 , 1) 

X_test_tensor = torch.tensor(X_test_scaled , dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values , dtype=torch.float32).view(-1 , 1) 

# Architecture:

- inputs :
 - x1- temp
 - x2-vaccum
 - x3- pressure
 - x4- humidity

 
- we will have 2 hidden layers , 6 neurons in each layer 

In [10]:
# DATA Loader creates the batches for us
# Data Loader also define how the data will be laded for the training 
# Tensor data set class is a class that helps the dataloader to access the data, becasue the dataloader is not cabable to access directly


# implementation"

In [13]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size=32)

### Building ANN Model:

In [15]:
# ANN model is trained in the form of class , which will be built on a base class nn.module

In [25]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
             #1st hidden:
             # nn.Linear (in_features, out_features) - syntax
             nn.Linear(X_train.shape[1], 6),
             nn.ReLU(),

         # 2nd hidden layer
             nn.Linear (6, 6),
             nn.ReLU(),

         #output layer
             nn.Linear(6,1),
        )

        def forward(self, x):
            return self.model(x)

       #pytorch will handel backward propagation automatically (autograd handels it)

In [26]:
import torch.optim as optim

model = ANN()

# loss, optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

### Training the model:

In [27]:
# we are going to have 100 epochs

train_losses = []

epochs = 100

for epoch in range(epochs):
    model.train()              # model adjusts the weights in the train mode
    running_loss = 0.0 #total training loss for 2 epoch

    for xb, yb in train_loader:
        # train loader will give us batch, xb = features of 1 batch and yb= labels of one batch

        optimizer.zero_grad()
        outputs = model(xb)   # predicted outputs for this batch
        loss = crietrion(outputs, yb)
        loss.backward()    #back propagation, computes the gradients
        optimizer.step() # params update

        running_loss += loss.item()  # loss is a tensor => py float

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


    # Validation:
    running_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs, yb)
            running_val_loss += loss
    

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)


    print(f"epoch ${epoch+1}/{epochs} ==> train loss = ${epoch_train_loss} & val loss =${epoch_val_loss}")

NotImplementedError: Module [ANN] is missing the required "forward" function